# Linearization


In [5]:
import sympy as sp

from lib.parameters import (
    FR_0,
    Q1_0,
    Q2_0,
    Q3_0,
    T0_0,
    T1_0,
    T2_0,
    T3_0,
    Ff1_0,
    Ff2_0,
    xA1_0,
    xA2_0,
    xA3_0,
    xB1_0,
    xB2_0,
    xB3_0,
)
from lib.sp_model import (
    FR,
    Q1,
    Q2,
    Q3,
    T0,
    T1,
    T2,
    T3,
    Ff1,
    Ff2,
    dT1dt,
    dT2dt,
    dT3dt,
    dxA1dt,
    dxA2dt,
    dxA3dt,
    dxB1dt,
    dxB2dt,
    dxB3dt,
    xA1,
    xA2,
    xA3,
    xB1,
    xB2,
    xB3,
)

f = sp.Matrix([dT1dt, dT2dt, dT3dt, dxA1dt, dxB1dt, dxA2dt, dxB2dt, dxA3dt, dxB3dt])
X = sp.Matrix([T1, T2, T3, xA1, xB1, xA2, xB2, xA3, xB3])
U = sp.Matrix([Ff1, Ff2, FR, Q1, Q2, Q3, T0])

U0_vals = {Ff1: Ff1_0, Ff2: Ff2_0, FR: FR_0, Q1: Q1_0, Q2: Q2_0, Q3: Q3_0, T0: T0_0}

X0_vals = {
    T1: T1_0,
    T2: T2_0,
    T3: T3_0,
    xA1: xA1_0,
    xB1: xB1_0,
    xA2: xA2_0,
    xB2: xB2_0,
    xA3: xA3_0,
    xB3: xB3_0,
}

A = f.jacobian(X).subs(X0_vals).subs(U0_vals)
B = f.jacobian(U).subs(X0_vals).subs(U0_vals)


# Model


In [6]:
from collections.abc import Callable

import numpy as np


def model(t: float, y: np.ndarray, u: list[Callable[[float], float]]):
    X = y
    U = np.array([ui(t) for ui in u]) - np.array(list(U0_vals.values()))

    return A @ X + B @ U


# Simulation


In [7]:
from scipy.integrate import solve_ivp

from lib.inputs import Ff1_func, Ff2_func, FR_func, Q1_func, Q2_func, Q3_func, T0_func
from lib.parameters import y0
from lib.plots import plot_system_inputs, plot_system_outputs
from lib.utils import input_names

base_funcs = [Ff1_func, Ff2_func, FR_func, Q1_func, Q2_func, Q3_func, T0_func]

t = np.linspace(0, 2.5, 1500)

for idx, name in enumerate(input_names):
    u = list(base_funcs)
    u[idx] = lambda t, f=base_funcs[idx]: f(t, disturb=True)

    sol = solve_ivp(
        model,
        [t[0], t[-1]],
        np.zeros_like(y0),
        t_eval=t,
        args=(u,),
        rtol=1e-10,
        atol=1e-10,
    )

    u_eval = [ui(t) for ui in u]
    soly = sol.y + np.array(list(X0_vals.values()))[:, None]
    plot_system_inputs(t, u_eval, f"linear/step-{name}-inputs")
    plot_system_outputs(t, soly, f"linear/step-{name}-outputs")


Plot saved to ../figures/linear/step-Ff1-inputs.png
Plot saved to ../figures/linear/step-Ff1-outputs.png
Plot saved to ../figures/linear/step-Ff2-inputs.png
Plot saved to ../figures/linear/step-Ff2-outputs.png
Plot saved to ../figures/linear/step-FR-inputs.png
Plot saved to ../figures/linear/step-FR-outputs.png
Plot saved to ../figures/linear/step-Q1-inputs.png
Plot saved to ../figures/linear/step-Q1-outputs.png
Plot saved to ../figures/linear/step-Q2-inputs.png
Plot saved to ../figures/linear/step-Q2-outputs.png
Plot saved to ../figures/linear/step-Q3-inputs.png
Plot saved to ../figures/linear/step-Q3-outputs.png
Plot saved to ../figures/linear/step-T0-inputs.png
Plot saved to ../figures/linear/step-T0-outputs.png


In [8]:
import matplotlib.pyplot as plt

from lib.np_model import model as nonlinear_model
from lib.plots import plot_or_show
from lib.utils import (
    input_labels,
    input_units,
    output_labels,
    output_names,
    output_units,
)


def T0_custom(t, disturb=False):
    if t < 0.5:
        return T0_0
    elif t < 2.0:
        return T0_0 + 5.0
    return T0_0 + 32.0


u = [Ff1_func, Ff2_func, FR_func, Q1_func, Q2_func, Q3_func, T0_custom]

t = np.linspace(0, 5, 3000)

sol_nl = solve_ivp(
    nonlinear_model, [t[0], t[-1]], y0, t_eval=t, args=(u,), rtol=1e-10, atol=1e-10
)
sol_lin = solve_ivp(
    model, [t[0], t[-1]], np.zeros_like(y0), t_eval=t, args=(u,), rtol=1e-10, atol=1e-10
)

Y_lin = sol_lin.y + np.array(list(X0_vals.values()))[:, None]

# --- Plotting ---
var_idx = 4
G_name = f"{output_names[var_idx]}_{input_names[6]}"
G_ylabel = f"{output_labels[var_idx]} / {output_units[6]}"

T0_plot = np.array([T0_custom(tt) for tt in t])
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(8, 6))

# Entrada
ax[0].plot(t, T0_plot)
ax[0].set_ylabel(f"{input_labels[6]} / {input_units[6]}")
ax[0].grid(True)

# xB1
ax[1].plot(sol_nl.t, sol_nl.y[var_idx], label="Não linear")
ax[1].plot(sol_lin.t, Y_lin[var_idx], "--", label="Linear")

ax[1].set_xlabel("Tempo / h")
ax[1].set_ylabel(G_ylabel)
ax[1].grid(True)
ax[1].legend()

plot_or_show(f"linear_vs_nonlinear/{G_name}")


Plot saved to ../figures/linear_vs_nonlinear/xB1_T0.png
